# HW03 深度学习作业
姓名：黄大康
学号：20234080423
日期：2026.06.03

## 2.1 卷积理论计算题
输入：$C_{in}=3, H=W=32$；卷积核：16个，$k_{ch}=3,k_h=k_w=5$，$P=2,S=2$
输出尺寸公式：$H_{out}=W_{out}=\lfloor\frac{H+2P-K}{S}+1\rfloor$
$$H_{out}=\frac{32+4-5}{2}+1=16.5\Rightarrow16$$
1. 输出特征图：**16×16×16**
2. 单个输出像素乘法次数 = $3\times5\times5=75$

## 2.2 手动实现MaxPool2d（numpy）

In [9]:
import numpy as np

def max_pool2d(x, kernel_size, stride, padding):
    N,C,H,W = x.shape
    kh,kw = kernel_size if isinstance(kernel_size,tuple) else (kernel_size,kernel_size)
    ph,pw = padding if isinstance(padding,tuple) else (padding,padding)
    sh,sw = stride if isinstance(stride,tuple) else (stride,stride)
    # padding填充
    pad_x = np.pad(x,((0,0),(0,0),(ph,ph),(pw,pw)),mode='constant')
    Ho = (H + 2*ph - kh)//sh +1
    Wo = (W + 2*pw - kw)//sw +1
    out = np.zeros((N,C,Ho,Wo))
    for i in range(Ho):
        for j in range(Wo):
            hs = i*sh; he = hs+kh
            ws = j*sw; we = ws+kw
            out[:,:,i,j] = np.max(pad_x[:,:,hs:he,ws:we],axis=(-2,-1))
    return out

# 测试
if __name__=='__main__':
    test = np.random.rand(2,3,8,8)
    res = max_pool2d(test,kernel_size=2,stride=2,padding=0)
    print("池化输出shape:",res.shape)

池化输出shape: (2, 3, 4, 4)


## 3.1 VGG卷积参数量计算（无bias，输入输出通道均C）
1. 单层5×5卷积：$C\times C\times5\times5=\boldsymbol{25C^2}$
2. 两层串联3×3卷积：$C\cdot C\cdot9 + C\cdot C\cdot9=\boldsymbol{18C^2}$

## 3.2 NiN Block实现

In [10]:
import torch
import torch.nn as nn

def nin_block(in_channels,out_channels,kernel_size,stride,padding):
    blk = nn.Sequential(
        nn.Conv2d(in_channels,out_channels,kernel_size,stride,padding),
        nn.ReLU(),
        nn.Conv2d(out_channels,out_channels,kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels,out_channels,kernel_size=1),
        nn.ReLU()
    )
    return blk

#测试
blk = nin_block(3,16,5,1,2)
x=torch.rand(1,3,32,32)
print("NiN输出shape:",blk(x).shape)

NiN输出shape: torch.Size([1, 16, 32, 32])


## 4.1 BN批量归一化计算
$x=[2,4,6,8],\gamma=2,\beta=1,\epsilon=0$
$\mu=5,\sigma^2=5$
$y=\gamma\frac{x-\mu}{\sqrt{\sigma^2}}+\beta$
$y_1=1-2/\sqrt5,\ y_2=1,\ y_3=3,\ y_4=1+2/\sqrt5$

## 4.2 ResNet残差块实现

In [15]:
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self,in_channels,out_channels,use_1x1conv=False,stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels,out_channels,kernel_size=3,padding=1,stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels,out_channels,kernel_size=3,padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = None
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels,out_channels,kernel_size=1,stride=stride)
    def forward(self,x):
        y = torch.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.conv3 is not None:
            x = self.conv3(x)
        return torch.relu(y+x)

#测试
res_blk = Residual(3,64,use_1x1conv=True,stride=2)
inp = torch.rand(1,3,32,32)
print("残差块输出shape:",res_blk(inp).shape)

残差块输出shape: torch.Size([1, 64, 16, 16])


## 5.1 微调理论
1. 底层预训练提取通用基础特征，参数收敛优良，小学习率/冻结防止破坏特征；顶层全新随机初始化，大学习率快速适配新数据集。
2. 数据少、数据集同源：冻结全部骨干网络参数，仅训练最后分类头，最大限度防止过拟合。

## 5.2 图像增广transform流水线

In [16]:
from torchvision import transforms

aug_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(size=224,scale=(0.08,1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5,contrast=0.5,saturation=0.5,hue=0),
    transforms.ToTensor()
])
print("图像增强管道构建完成")

图像增强管道构建完成


## 6.1 IoU计算
A=[10,10,50,50],B=[30,30,70,70]
交集：[30,30,50,50]，面积=400
A面积=1600，B面积=1600，并集=2800
$IoU=400/2800=\boldsymbol{1/7\approx0.1429}$

## 6.2 标签平滑交叉熵实现($\epsilon=0.1$)

In [14]:
import torch
import torch.nn.functional as F

def label_smooth_ce(logits,labels,K,eps=0.1):
    batch = logits.shape[0]
    smooth_target = torch.full((batch,K),eps/(K-1),device=logits.device)
    smooth_target.scatter_(1,labels.unsqueeze(1),1-eps)
    log_p = F.log_softmax(logits,dim=-1)
    loss = -(smooth_target*log_p).sum(dim=1).mean()
    return loss

#测试
logits = torch.randn(4,5)
lab = torch.tensor([0,2,1,3])
loss = label_smooth_ce(logits,lab,K=5,eps=0.1)
print("标签平滑损失值:",loss.item())

标签平滑损失值: 1.7336130142211914
